# 🐀 NYC Rat Reservoir — Early Detection Pipeline
**Datasets:** DOHMH Restaurant Inspections + 311 Rat Sightings + Rodent Inspections + DOB Construction Permits

**Join Strategy:** Zip-code aggregation, spatial proximity (BallTree), and temporal windowing

**Output:** A "Heat Score" (0–100) per zip code identifying Rat Reservoir zones

## Setup & Install

In [ ]:
!pip install pandas sodapy scikit-learn geopandas shapely folium -q

In [ ]:
import pandas as pd
import numpy as np
from sodapy import Socrata
from sklearn.neighbors import BallTree
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

print("All imports loaded ✓")

---
## 1. Download Datasets from NYC Open Data

In [ ]:
# Connect to NYC Open Data Socrata API (no auth needed for public data)
client = Socrata("data.cityofnewyork.us", None)  # Replace None with app_token for higher rate limits

In [ ]:
# 1a. Restaurant Inspections — rodent-related violations only
# Violation codes: 04K=rats, 04L=mice, 04M=roaches, 04N=filth flies, 08A=pest conditions
print("Downloading Restaurant Inspections (rodent violations)...")
restaurant_results = client.get(
    "43nn-pn8j",
    where="violation_code IN ('04K', '04L', '04M', '04N', '08A')",
    limit=500000,
    order="inspection_date DESC"
)
df_restaurants = pd.DataFrame.from_records(restaurant_results)
print(f"✓ {len(df_restaurants):,} rodent-related restaurant violations")
df_restaurants.head()

In [ ]:
# 1b. 311 Rat Sightings
print("Downloading 311 Rat Sightings...")
rat_results = client.get(
    "3q43-55fe",
    limit=500000,
    order="created_date DESC"
)
df_rats = pd.DataFrame.from_records(rat_results)
print(f"✓ {len(df_rats):,} rat sighting complaints")
df_rats.head()

In [ ]:
# 1c. DOHMH Rodent Inspections
print("Downloading DOHMH Rodent Inspections...")
rodent_insp_results = client.get(
    "p937-wjvj",
    limit=500000,
    order="inspection_date DESC"
)
df_rodent_insp = pd.DataFrame.from_records(rodent_insp_results)
print(f"✓ {len(df_rodent_insp):,} rodent inspection records")
df_rodent_insp.head()

In [ ]:
# 1d. DOB Construction Permits
print("Downloading DOB Construction Permits...")
permits_results = client.get(
    "rbx6-tga4",
    limit=500000,
    order="approved_date DESC"
)
df_permits = pd.DataFrame.from_records(permits_results)
print(f"✓ {len(df_permits):,} construction permit records")
df_permits.head()

---
## 2. Clean & Standardize Each Dataset

In [ ]:
# 2a. Clean Restaurant Inspections
df_restaurants["inspection_date"] = pd.to_datetime(df_restaurants["inspection_date"], errors="coerce")
df_restaurants["latitude"] = pd.to_numeric(df_restaurants.get("latitude"), errors="coerce")
df_restaurants["longitude"] = pd.to_numeric(df_restaurants.get("longitude"), errors="coerce")
df_restaurants["zipcode"] = df_restaurants.get("zipcode", "").astype(str).str.strip()
df_restaurants["score"] = pd.to_numeric(df_restaurants.get("score"), errors="coerce")

restaurant_cols = [
    "camis", "dba", "boro", "zipcode", "street", "building",
    "latitude", "longitude", "inspection_date", "violation_code",
    "violation_description", "critical_flag", "score", "grade"
]
df_restaurants = df_restaurants[[c for c in restaurant_cols if c in df_restaurants.columns]]
df_restaurants = df_restaurants.dropna(subset=["latitude", "longitude"])

print(f"✓ Restaurants cleaned: {len(df_restaurants):,} rows")
df_restaurants.info()

In [ ]:
# 2b. Clean Rat Sightings
df_rats["created_date"] = pd.to_datetime(df_rats.get("created_date"), errors="coerce")
df_rats["latitude"] = pd.to_numeric(df_rats.get("latitude"), errors="coerce")
df_rats["longitude"] = pd.to_numeric(df_rats.get("longitude"), errors="coerce")
df_rats["incident_zip"] = df_rats.get("incident_zip", "").astype(str).str[:5]

rat_cols = [
    "unique_key", "created_date", "incident_zip", "incident_address",
    "city", "borough", "latitude", "longitude",
    "descriptor", "location_type", "status"
]
df_rats = df_rats[[c for c in rat_cols if c in df_rats.columns]]
df_rats = df_rats.dropna(subset=["latitude", "longitude"])

print(f"✓ Rat sightings cleaned: {len(df_rats):,} rows")
df_rats.info()

In [ ]:
# 2c. Clean Rodent Inspections
df_rodent_insp["inspection_date"] = pd.to_datetime(df_rodent_insp.get("inspection_date"), errors="coerce")
df_rodent_insp["latitude"] = pd.to_numeric(df_rodent_insp.get("latitude"), errors="coerce")
df_rodent_insp["longitude"] = pd.to_numeric(df_rodent_insp.get("longitude"), errors="coerce")
df_rodent_insp["zip_code"] = df_rodent_insp.get("zip_code", "").astype(str).str[:5]

rodent_cols = [
    "job_id", "inspection_date", "result", "zip_code", "borough_code",
    "latitude", "longitude", "community_district", "inspection_type"
]
df_rodent_insp = df_rodent_insp[[c for c in rodent_cols if c in df_rodent_insp.columns]]
df_rodent_insp = df_rodent_insp.dropna(subset=["latitude", "longitude"])

print(f"✓ Rodent inspections cleaned: {len(df_rodent_insp):,} rows")
df_rodent_insp.info()

In [ ]:
# 2d. Clean Construction Permits
df_permits["approved_date"] = pd.to_datetime(df_permits.get("approved_date"), errors="coerce")

for lat_col in ["latitude", "gis_latitude", "lat"]:
    if lat_col in df_permits.columns:
        df_permits["latitude"] = pd.to_numeric(df_permits[lat_col], errors="coerce")
        break

for lon_col in ["longitude", "gis_longitude", "lng"]:
    if lon_col in df_permits.columns:
        df_permits["longitude"] = pd.to_numeric(df_permits[lon_col], errors="coerce")
        break

for zip_col in ["postcode", "zip", "zipcode"]:
    if zip_col in df_permits.columns:
        df_permits["zipcode"] = df_permits[zip_col].astype(str).str[:5]
        break

permit_cols = ["job_filing_number", "approved_date", "latitude", "longitude", "zipcode",
               "borough", "job_type", "work_type"]
df_permits = df_permits[[c for c in permit_cols if c in df_permits.columns]]
if "latitude" in df_permits.columns:
    df_permits = df_permits.dropna(subset=["latitude", "longitude"])

print(f"✓ Construction permits cleaned: {len(df_permits):,} rows")
df_permits.info()

---
## 3A. Join Method A — Zip Code Aggregation
Aggregate all signals per zip code → primary table for Heat Score

In [ ]:
# Aggregate restaurant rodent violations per zip
rest_by_zip = df_restaurants.groupby("zipcode").agg(
    restaurant_rodent_violations=("violation_code", "count"),
    restaurant_critical_count=("critical_flag", lambda x: (x == "Critical").sum()),
    restaurant_avg_score=("score", "mean"),
    restaurant_unique_locations=("camis", "nunique"),
    restaurant_latest_violation=("inspection_date", "max"),
).reset_index().rename(columns={"zipcode": "zip_code"})

print(f"Restaurant violations aggregated: {len(rest_by_zip)} zip codes")
rest_by_zip.head()

In [ ]:
# Aggregate 311 rat sightings per zip
rat_count_col = "unique_key" if "unique_key" in df_rats.columns else "created_date"
rats_by_zip = df_rats.groupby("incident_zip").agg(
    rat_sighting_count=(rat_count_col, "count"),
    rat_latest_sighting=("created_date", "max"),
    rat_earliest_sighting=("created_date", "min"),
).reset_index().rename(columns={"incident_zip": "zip_code"})

print(f"Rat sightings aggregated: {len(rats_by_zip)} zip codes")
rats_by_zip.head()

In [ ]:
# Aggregate rodent inspections per zip
if "zip_code" in df_rodent_insp.columns and len(df_rodent_insp) > 0:
    rodent_by_zip = df_rodent_insp.groupby("zip_code").agg(
        rodent_inspection_count=("inspection_date", "count"),
        rodent_active_rat_signs=("result", lambda x: x.str.contains("Rat Activity", case=False, na=False).sum()),
        rodent_latest_inspection=("inspection_date", "max"),
    ).reset_index()
    print(f"Rodent inspections aggregated: {len(rodent_by_zip)} zip codes")
else:
    rodent_by_zip = pd.DataFrame(columns=["zip_code"])
    print("Rodent inspections: no zip_code column found, skipping")

rodent_by_zip.head()

In [ ]:
# Aggregate construction permits per zip
if "zipcode" in df_permits.columns and len(df_permits) > 0:
    permits_by_zip = df_permits.groupby("zipcode").agg(
        construction_permit_count=("approved_date", "count"),
        construction_latest_permit=("approved_date", "max"),
    ).reset_index().rename(columns={"zipcode": "zip_code"})
    print(f"Construction permits aggregated: {len(permits_by_zip)} zip codes")
else:
    permits_by_zip = pd.DataFrame(columns=["zip_code"])
    print("Construction permits: no zipcode column found, skipping")

permits_by_zip.head()

In [ ]:
# MERGE ALL ON ZIP CODE
df_merged = rest_by_zip.merge(rats_by_zip, on="zip_code", how="outer")
df_merged = df_merged.merge(rodent_by_zip, on="zip_code", how="outer")
df_merged = df_merged.merge(permits_by_zip, on="zip_code", how="outer")

# Fill NaN counts with 0
count_cols = [c for c in df_merged.columns if any(k in c for k in ["count", "violations", "signs", "locations"])]
df_merged[count_cols] = df_merged[count_cols].fillna(0)

print(f"\n✓ Merged dataset: {len(df_merged)} zip codes × {len(df_merged.columns)} features")
df_merged.head(10)

---
## 3B. Join Method B — Spatial Proximity (Restaurant ↔ Nearby Rat Sightings)
Uses BallTree (haversine) to count rat sightings within 200m of each restaurant

In [ ]:
RADIUS_METERS = 200  # ~1 city block
EARTH_RADIUS_M = 6_371_000

def spatial_count_within_radius(target_df, source_df, target_lat, target_lon,
                                 source_lat, source_lon, radius_m=RADIUS_METERS):
    """Count how many source points fall within radius of each target point."""
    target_coords = np.radians(target_df[[target_lat, target_lon]].values)
    source_coords = np.radians(source_df[[source_lat, source_lon]].values)
    tree = BallTree(source_coords, metric="haversine")
    radius_rad = radius_m / EARTH_RADIUS_M
    counts = tree.query_radius(target_coords, r=radius_rad, count_only=True)
    return counts

print("Spatial join function ready ✓")

In [ ]:
# Get unique restaurants (one row per location)
SAMPLE_SIZE = 50_000
rest_sample = df_restaurants.drop_duplicates(subset=["camis"]).head(SAMPLE_SIZE).copy()

if len(rest_sample) > 0 and len(df_rats) > 0:
    print(f"Computing nearby rat sightings for {len(rest_sample):,} restaurants (radius={RADIUS_METERS}m)...")
    rest_sample["nearby_rat_sightings"] = spatial_count_within_radius(
        rest_sample, df_rats, "latitude", "longitude", "latitude", "longitude", radius_m=RADIUS_METERS
    )
    
    if len(df_permits) > 0 and "latitude" in df_permits.columns:
        print(f"Computing nearby construction permits (radius=300m)...")
        rest_sample["nearby_construction"] = spatial_count_within_radius(
            rest_sample, df_permits, "latitude", "longitude", "latitude", "longitude", radius_m=300
        )
    
    print(f"\n✓ Avg nearby rat sightings per restaurant: {rest_sample['nearby_rat_sightings'].mean():.1f}")
    rest_sample.sort_values("nearby_rat_sightings", ascending=False).head(10)
else:
    print("Insufficient data for spatial join")

---
## 3C. Join Method C — Temporal Window (±90 Days)
For each restaurant violation, count rat sightings in the same zip within ±90 days

In [ ]:
TIME_WINDOW_DAYS = 90

rest_temporal = df_restaurants.dropna(subset=["inspection_date", "zipcode"]).head(20_000).copy()

if len(rest_temporal) > 0 and len(df_rats) > 0:
    # Index rat sightings by zip for fast lookup
    rats_indexed = df_rats.dropna(subset=["created_date"]).copy()
    rats_by_zip_dict = {
        z: group["created_date"].values
        for z, group in rats_indexed.groupby("incident_zip")
    }
    
    def count_rats_in_window(row):
        zipcode = str(row["zipcode"])
        insp_date = row["inspection_date"]
        if zipcode not in rats_by_zip_dict or pd.isna(insp_date):
            return 0
        rat_dates = pd.to_datetime(rats_by_zip_dict[zipcode])
        window_start = insp_date - timedelta(days=TIME_WINDOW_DAYS)
        window_end = insp_date + timedelta(days=TIME_WINDOW_DAYS)
        return int(((rat_dates >= window_start) & (rat_dates <= window_end)).sum())
    
    print(f"Computing temporal join for {len(rest_temporal):,} violations (±{TIME_WINDOW_DAYS} days)...")
    rest_temporal["rats_within_90days"] = rest_temporal.apply(count_rats_in_window, axis=1)
    
    print(f"\n✓ Avg rat sightings within ±90 days (same zip): {rest_temporal['rats_within_90days'].mean():.1f}")
    rest_temporal.sort_values("rats_within_90days", ascending=False).head(10)
else:
    print("Insufficient data for temporal join")

---
## 4. Compute the Heat Score

In [ ]:
# Features and weights for the composite Heat Score
score_features = [
    "restaurant_rodent_violations",
    "restaurant_critical_count",
    "rat_sighting_count",
    "rodent_active_rat_signs",
    "rodent_inspection_count",
    "construction_permit_count",
]

weights = {
    "restaurant_rodent_violations": 0.25,
    "restaurant_critical_count":    0.15,
    "rat_sighting_count":           0.30,   # Heaviest — real-time citizen reports
    "rodent_active_rat_signs":      0.15,
    "rodent_inspection_count":      0.05,
    "construction_permit_count":    0.10,   # Construction disturbs rat nests
}

available_features = [f for f in score_features if f in df_merged.columns]
df_score = df_merged[["zip_code"] + available_features].copy()
df_score[available_features] = df_score[available_features].fillna(0)

# Normalize 0–1
scaler = MinMaxScaler()
df_score[available_features] = scaler.fit_transform(df_score[available_features])

# Weighted sum → scale to 0–100
df_score["heat_score"] = sum(df_score[f] * weights.get(f, 0.1) for f in available_features)
df_score["heat_score"] = (df_score["heat_score"] / df_score["heat_score"].max() * 100).round(1)

# Merge back
df_final = df_merged.merge(df_score[["zip_code", "heat_score"]], on="zip_code", how="left")
df_final = df_final.sort_values("heat_score", ascending=False)

print("✓ Heat Scores computed\n")
print("Top 15 Highest-Risk Zip Codes:")
df_final[["zip_code", "heat_score", "restaurant_rodent_violations", "rat_sighting_count",
          "rodent_active_rat_signs", "construction_permit_count"]].head(15)

---
## 5. Visualize — Heat Score Map

In [ ]:
import folium
from folium.plugins import HeatMap

# Get average lat/lon per zip from rat sightings (most coverage)
zip_coords = df_rats.groupby("incident_zip").agg(
    lat=("latitude", "mean"),
    lon=("longitude", "mean")
).reset_index().rename(columns={"incident_zip": "zip_code"})

df_map = df_final.merge(zip_coords, on="zip_code", how="inner")
df_map = df_map.dropna(subset=["lat", "lon", "heat_score"])

# Create folium map centered on NYC
m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles="CartoDB dark_matter")

# Add heatmap layer
heat_data = df_map[["lat", "lon", "heat_score"]].values.tolist()
HeatMap(
    heat_data,
    min_opacity=0.4,
    radius=20,
    blur=15,
    max_zoom=13,
    gradient={0.2: "blue", 0.4: "lime", 0.6: "yellow", 0.8: "orange", 1.0: "red"}
).add_to(m)

# Add circle markers for top 20 hotspots
for _, row in df_map.head(20).iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=row["heat_score"] / 10,
        color="red",
        fill=True,
        fill_opacity=0.7,
        popup=f"ZIP: {row['zip_code']}<br>Heat Score: {row['heat_score']}<br>"
               f"Rat Sightings: {int(row.get('rat_sighting_count', 0))}<br>"
               f"Restaurant Violations: {int(row.get('restaurant_rodent_violations', 0))}"
    ).add_to(m)

print("✓ Map generated — scroll down to view")
m

---
## 6. Quick EDA — Distribution & Correlations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("NYC Rat Reservoir — Feature Distributions", fontsize=16, fontweight="bold")

# Heat Score distribution
axes[0, 0].hist(df_final["heat_score"].dropna(), bins=30, color="#e74c3c", edgecolor="white")
axes[0, 0].set_title("Heat Score Distribution")
axes[0, 0].set_xlabel("Heat Score (0-100)")

# Rat sightings vs restaurant violations
axes[0, 1].scatter(
    df_final["rat_sighting_count"], df_final["restaurant_rodent_violations"],
    alpha=0.5, c=df_final["heat_score"], cmap="YlOrRd", s=20
)
axes[0, 1].set_title("Rat Sightings vs Restaurant Violations")
axes[0, 1].set_xlabel("311 Rat Sightings")
axes[0, 1].set_ylabel("Restaurant Rodent Violations")

# Top 15 zip codes bar chart
top15 = df_final.head(15)
axes[1, 0].barh(top15["zip_code"].astype(str), top15["heat_score"], color="#e74c3c")
axes[1, 0].set_title("Top 15 Zip Codes by Heat Score")
axes[1, 0].set_xlabel("Heat Score")
axes[1, 0].invert_yaxis()

# Construction vs rat sightings
if "construction_permit_count" in df_final.columns:
    axes[1, 1].scatter(
        df_final["construction_permit_count"], df_final["rat_sighting_count"],
        alpha=0.5, color="#3498db", s=20
    )
    axes[1, 1].set_title("Construction Activity vs Rat Sightings")
    axes[1, 1].set_xlabel("Construction Permits")
    axes[1, 1].set_ylabel("Rat Sightings")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr_cols = [c for c in available_features + ["heat_score"] if c in df_final.columns]
corr = df_final[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdYlBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in corr_cols], rotation=45, ha="right", fontsize=9)
ax.set_yticklabels([c.replace("_", " ") for c in corr_cols], fontsize=9)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im)
plt.title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 7. Save Outputs

In [ ]:
# Save all outputs
df_final.to_csv("rat_reservoir_heat_scores_by_zip.csv", index=False)
print(f"✓ rat_reservoir_heat_scores_by_zip.csv  ({len(df_final)} rows)")

if "nearby_rat_sightings" in rest_sample.columns:
    rest_sample.to_csv("restaurant_spatial_risk.csv", index=False)
    print(f"✓ restaurant_spatial_risk.csv            ({len(rest_sample)} rows)")

if "rats_within_90days" in rest_temporal.columns:
    rest_temporal.to_csv("restaurant_temporal_risk.csv", index=False)
    print(f"✓ restaurant_temporal_risk.csv            ({len(rest_temporal)} rows)")

df_restaurants.to_csv("cleaned_restaurant_violations.csv", index=False)
df_rats.to_csv("cleaned_rat_sightings.csv", index=False)
df_rodent_insp.to_csv("cleaned_rodent_inspections.csv", index=False)
df_permits.to_csv("cleaned_construction_permits.csv", index=False)
print("✓ All cleaned datasets saved")

print("\n" + "="*50)
print("DONE ✓")
print("="*50)